## Структура датасета CISI

`CISI.ALL` содержит 1460 документов: `.I` — ID, `.T` — заголовок, `.A` — авторы, `.B` — библиографические сведения, `.W` — аннотация, `.X` — перекрёстные ссылки.
Тройки `.X` сохраняются без интерпретации второго столбца как числа цитирований: такой смысл не подтверждён описанием локальных данных.

`CISI.QRY` содержит 112 запросов; текст запроса находится прежде всего в `.W`, некоторые записи содержат дополнительные поля.

`CISI.REL` содержит четыре столбца. Первые два задают ID запроса и релевантного документа; последние два не используются как оценка релевантности. Нули в них не означают нерелевантность. Не все 112 запросов имеют разметку: количество проверяется при загрузке.

In [ ]:
from pathlib import Path

DATA_DIR = next((p for p in (Path("notebook"), Path("."))
                 if all((p / name).is_file() for name in ("CISI.ALL", "CISI.QRY", "CISI.REL"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run this notebook from the repository root or the notebook directory.")

def data_path(filename):
    return DATA_DIR / filename

def parse(filename):
    records = {}
    record = None
    field = None
    fields = {".T": "title", ".A": "author", ".B": "year", ".W": "text", ".X": "x_ref"}
    with open(filename, encoding="utf-8") as stream:
        for number, raw in enumerate(stream, 1):
            line = raw.strip()
            if not line:
                continue
            if line.startswith(".I "):
                doc_id = int(line.split()[1])
                if doc_id in records:
                    raise ValueError(f"Duplicate ID {doc_id} in {filename}:{number}")
                record = {"id": doc_id, "title": "", "author": [], "year": "", "text": "", "x_ref": []}
                records[doc_id] = record
                field = None
            elif line.startswith(".") and len(line.split()[0]) == 2:
                field = fields.get(line.split()[0])
            elif record is not None and field is not None:
                if field == "x_ref":
                    values = tuple(map(int, line.split()))
                    if len(values) != 3:
                        raise ValueError(f"Invalid cross-reference in {filename}:{number}")
                    record[field].append(values)
                elif field == "author":
                    record[field].append(line)
                else:
                    record[field] = (record[field] + " " + line).strip()
    return records

In [ ]:
documents = parse(data_path("CISI.ALL"))
queries = parse(data_path("CISI.QRY"))
print(documents[1])
print(queries[1])

In [ ]:
from collections import defaultdict

relevance = defaultdict(set)
with open(data_path("CISI.REL"), encoding="utf-8") as stream:
    for number, line in enumerate(stream, 1):
        values = line.split()
        if not values:
            continue
        query_id, doc_id = map(int, values[:2])
        if query_id not in queries or doc_id not in documents:
            raise ValueError(f"Unknown query/document ID in CISI.REL:{number}")
        relevance[query_id].add(doc_id)
relevance = dict(relevance)
print(sorted(relevance[1]))
print(f"Queries with relevance judgments: {len(relevance)} / {len(queries)}")

In [ ]:
print(f"Документов: {len(documents)}")
print(f"Запросов:   {len(queries)}")
print(f"Тип запроса №1: {type(queries[1])}")
print(f"Запрос №1: {queries[1]}")

## Оценка качества

Ранжированная выдача сравнивается со всем множеством релевантных документов запроса из `CISI.REL`, а не с первыми K строками файла.

- Precision@K = число релевантных документов в первых K позициях / K; короткая выдача не уменьшает знаменатель.
- Recall@K = число релевантных документов в первых K позициях / число всех релевантных документов запроса.
- AP = сумма Precision на позициях релевантных документов / число всех релевантных документов запроса.
- MAP = средний AP по запросам с разметкой. Запросы без разметки не считаются нерелевантными и исключаются из оценки.

Булев поиск оценивается как множество с Precision и Recall, поскольку у него нет ранжирования.

In [ ]:
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

def ensure_nltk_resource(resource_path, package_name):
    for candidate in (resource_path, resource_path + ".zip"):
        try:
            nltk.data.find(candidate)
            return
        except LookupError:
            pass
    if not nltk.download(package_name, quiet=True):
        raise RuntimeError(f"Missing NLTK resource: {package_name}. Download it and rerun the notebook.")

ensure_nltk_resource("tokenizers/punkt_tab", "punkt_tab")
ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")
wordnet.ensure_loaded()
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(text):
    word_tokens = word_tokenize(text.lower())
    filtered = [word for word in word_tokens 
                if word not in stop_words and (word.isalpha() or word.isdigit())]
    return filtered

for doc_id, doc in documents.items():
    doc["raw_words"] = [w for w in word_tokenize((doc["title"] + " " + doc["text"] + " " + " ".join(doc["author"])).lower()) if w.isalpha() or w.isdigit()]
    full_text = doc["title"] + " " + doc["text"]

    if doc["author"]:
        authors_text = " ".join(doc["author"])
        full_text = full_text + " " + authors_text

    # Заголовок
    title_tokens = clean_text(doc["title"])
    doc["title_tokens"] = title_tokens
    doc["title_stemmed"] = [stemmer.stem(w) for w in title_tokens]
    doc["title_lemmatized"] = [lemmatizer.lemmatize(w) for w in title_tokens]   

    # Авторы
    author_text = " ".join(doc["author"]) if doc["author"] else ""
    author_tokens = clean_text(author_text)
    doc["author_tokens"] = author_tokens
    doc["author_stemmed"] = [stemmer.stem(w) for w in author_tokens]
    doc["author_lemmatized"] = [lemmatizer.lemmatize(w) for w in author_tokens]
    
    # Основной текст
    text_tokens = clean_text(doc["text"])
    doc["text_tokens"] = text_tokens
    doc["text_stemmed"] = [stemmer.stem(w) for w in text_tokens]
    doc["text_lemmatized"] = [lemmatizer.lemmatize(w) for w in text_tokens]

    # Весь текст сразу
    tokens = clean_text(full_text)
    doc["tokens"] = tokens
    doc["stemmed_words"] = [stemmer.stem(word) for word in tokens]
    doc["lemmatized_words"] = [lemmatizer.lemmatize(word) for word in tokens]


for query_id, query in queries.items():
    query["raw_words"] = [w for w in word_tokenize((query["title"] + " " + query["text"] + " " + " ".join(query["author"])).lower()) if w.isalpha() or w.isdigit()]
    full_text = query["title"] + " " + query["text"]

    if query["author"]:
        authors_text = " ".join(query["author"])
        full_text = full_text + " " + authors_text

    # Заголовок
    title_tokens = clean_text(query["title"])
    query["title_tokens"] = title_tokens
    query["title_stemmed"] = [stemmer.stem(w) for w in title_tokens]
    query["title_lemmatized"] = [lemmatizer.lemmatize(w) for w in title_tokens]
    
    # Авторы
    author_text = " ".join(query["author"]) if query["author"] else ""
    author_tokens = clean_text(author_text)
    query["author_tokens"] = author_tokens
    query["author_stemmed"] = [stemmer.stem(w) for w in author_tokens]
    query["author_lemmatized"] = [lemmatizer.lemmatize(w) for w in author_tokens]
    
    # Основной текст
    text_tokens = clean_text(query["text"])
    query["text_tokens"] = text_tokens
    query["text_stemmed"] = [stemmer.stem(w) for w in text_tokens]
    query["text_lemmatized"] = [lemmatizer.lemmatize(w) for w in text_tokens]

    # Весь текст сразу
    tokens = clean_text(full_text)
    query["tokens"] = tokens
    query["stemmed_words"] = [stemmer.stem(word) for word in tokens]
    query["lemmatized_words"] = [lemmatizer.lemmatize(word) for word in tokens]

In [ ]:
import pandas as pd

raw_vocab = set()
cleaned_vocab = set()
stemmed_vocab = set()
lemmatized_vocab = set()

for doc_id, doc in documents.items():
    raw_text = (doc["title"] + " " + doc["text"] + " " + " ".join(doc["author"])).lower()
    raw_tokens = word_tokenize(raw_text.lower())
    raw_vocab.update(raw_tokens)
    
    cleaned_vocab.update(doc["tokens"])
    stemmed_vocab.update(doc["stemmed_words"])
    lemmatized_vocab.update(doc["lemmatized_words"])

data = {
    "Метод обработки CISI.ALL": [
        "Сырой текст (токены)",
        "После очистки (без стоп-слов и пунктуации)",
        "После стемминга (Porter Stemmer)",
        "После лемматизации (WordNet Lemmatizer)"
    ],
    "Уникальных токенов": [
        len(raw_vocab),
        len(cleaned_vocab),
        len(stemmed_vocab),
        len(lemmatized_vocab)
    ],
    "% от сырого текста": [
        100.0,
        round(len(cleaned_vocab) / len(raw_vocab) * 100, 1),
        round(len(stemmed_vocab) / len(raw_vocab) * 100, 1),
        round(len(lemmatized_vocab) / len(raw_vocab) * 100, 1)
    ],
    "Сокращение размера словаря": [
        0,
        len(raw_vocab) - len(cleaned_vocab),
        len(raw_vocab) - len(stemmed_vocab),
        len(raw_vocab) - len(lemmatized_vocab)
    ]
}

df = pd.DataFrame(data)
vocab_documents = df
display(df)

In [ ]:
raw_vocab = set()
cleaned_vocab = set()
stemmed_vocab = set()
lemmatized_vocab = set()

for query_id, query in queries.items():
    raw_text = (query["title"] + " " + query["text"] + " " + " ".join(query["author"])).lower()
    raw_tokens = word_tokenize(raw_text.lower())
    raw_vocab.update(raw_tokens)
    
    cleaned_vocab.update(query["tokens"])
    stemmed_vocab.update(query["stemmed_words"])
    lemmatized_vocab.update(query["lemmatized_words"])

data = {
    "Метод обработки CISI.QRY": [
        "Сырой текст",
        "После очистки",
        "После стемминга",
        "После лемматизации"
    ],
    "Уникальных токенов": [
        len(raw_vocab),
        len(cleaned_vocab),
        len(stemmed_vocab),
        len(lemmatized_vocab)
    ],
    "% от сырого текста": [
        100.0,
        round(len(cleaned_vocab) / len(raw_vocab) * 100, 1),
        round(len(stemmed_vocab) / len(raw_vocab) * 100, 1),
        round(len(lemmatized_vocab) / len(raw_vocab) * 100, 1)
    ],
    "Сокращение размера словаря": [
        0,
        len(raw_vocab) - len(cleaned_vocab),
        len(raw_vocab) - len(stemmed_vocab),
        len(raw_vocab) - len(lemmatized_vocab)
    ]
}

df = pd.DataFrame(data)
vocab_queries = df
display(df)

## Сравнительный анализ словаря

Таблицы выше вычисляются заново по всей коллекции и всем запросам. Размер словаря — число уникальных токенов, а не суммарное число их вхождений. Сокращение словаря само по себе не доказывает повышение качества поиска.

Очистка удаляет стоп-слова и токены, которые не состоят только из букв либо цифр; стемминг объединяет словоформы по правилам Porter. WordNetLemmatizer здесь использует часть речи `noun` по умолчанию, без контекстного морфологического анализа. Стоп-слова проверяются отдельно через индекс `raw_words`, в котором они сохранены.

## Построение обратного индекса

In [ ]:
from collections import defaultdict
import time
import math

index_timings = []

def build_simple_inverted_index(documents, field):
    start = time.perf_counter()
    index = defaultdict(set)
    for doc_id, doc in documents.items():
        for word in doc[field]:
            index[word].add(doc_id)
    result = {word: sorted(ids) for word, ids in index.items()}
    index_timings.append({"field": field, "type": "simple", "seconds": time.perf_counter() - start, "terms": len(result)})
    return result

def build_rich_inverted_index(documents, field):
    start = time.perf_counter()
    N = len(documents)
    index = defaultdict(dict)
    for doc_id, doc in documents.items():
        for word in doc[field]:
            meta = index[word].setdefault(doc_id, {"tf": 0})
            meta["tf"] += 1
    df = {word: len(postings) for word, postings in index.items()}
    lengths = {doc_id: len(doc[field]) for doc_id, doc in documents.items()}
    result = {
        "inverted_index": dict(index), "df": df,
        "idf": {word: math.log(N / count) for word, count in df.items()},
        "bm25_idf": {word: math.log1p((N - count + 0.5) / (count + 0.5)) for word, count in df.items()},
        "field": field, "N": N, "doc_lengths": lengths,
        "avg_dl": sum(lengths.values()) / N if N else 0.0,
    }
    index_timings.append({"field": field, "type": "rich", "seconds": time.perf_counter() - start, "terms": len(index)})
    return result

fields = [
    "raw_words", "tokens", "stemmed_words", "lemmatized_words",
    "title_tokens", "title_stemmed", "title_lemmatized",
    "author_tokens", "author_stemmed", "author_lemmatized",
    "text_tokens", "text_stemmed", "text_lemmatized",
]
simple_indexes = {field: build_simple_inverted_index(documents, field) for field in fields}
rich_indexes = {field: build_rich_inverted_index(documents, field) for field in fields}
display(pd.DataFrame(index_timings))

def query_field_for(index_field):
    if index_field in ("raw_words", "tokens", "stemmed_words", "lemmatized_words"):
        return index_field
    return {"tokens": "tokens", "stemmed": "stemmed_words", "lemmatized": "lemmatized_words"}[index_field.rsplit("_", 1)[1]]

## Время построения индексов

Измеряется полный вызов построения, включая сортировку списков, DF, IDF и длины документов. Это одиночные измерения на текущем компьютере: устойчивые выводы о скорости требуют повторов и сравнения медиан. Уменьшение числа терминов не гарантирует ускорение каждого запуска.

## Реализация алгоритмов ранжирования

In [ ]:
def boolean_search(query, query_field, index_field, operator="OR", fallback_field=None):
    if operator not in ("AND", "OR"):
        raise ValueError("operator must be AND or OR")
    words = query[query_field]
    sets = [set(simple_indexes[index_field].get(word, [])) for word in words]
    result = (set.intersection(*sets) if operator == "AND" else set.union(*sets)) if sets else set()
    if not result and fallback_field is not None:
        return boolean_search(query, query_field_for(fallback_field), fallback_field, operator)
    return result

In [ ]:
# Тест
query_id = 1
query = queries[query_id]

results = boolean_search(query, "stemmed_words", "stemmed_words", "OR")

print(f"Запрос №{query_id}: {query['text'][:70]}...")
print(f"Найдено документов: {len(results)}")
print(f"Первые 10: {sorted(results)[:10]}")

if query_id in relevance:
    relevant_set = set(relevance[query_id])
    print(f"\nРелевантных документов: {len(relevant_set)}")
    print(f"Пересечение: {len(results & relevant_set)}")

## Булев поиск без ранжирования

OR возвращает объединение списков документов, AND — пересечение. Число найденных документов и пересечение с разметкой запроса №1 вычислены выше. Высокая полнота одного запроса не гарантирует Recall = 100% на остальных запросах. Сортировка ID используется только для показа; она не означает сортировку по релевантности.

In [ ]:
def tfidf_search(query, query_field, rich_idx, normalize=True, fallback_idx=None):
    scores = defaultdict(float)
    for word in query[query_field]:
        weight = rich_idx["idf"].get(word, 0.0)
        if weight <= 0:
            continue
        for doc_id, meta in rich_idx["inverted_index"].get(word, {}).items():
            scores[doc_id] += meta["tf"] * weight
    if not scores and fallback_idx is not None:
        results = tfidf_search(query, query_field_for(fallback_idx["field"]), fallback_idx, normalize)
        return [(doc_id, score * 0.5) for doc_id, score in results]
    if normalize:
        for doc_id in scores:
            scores[doc_id] /= rich_idx["doc_lengths"][doc_id]
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))

In [ ]:
query_id = 1
query = queries[query_id]
field = "stemmed_words"
idx = rich_indexes[field]

# TF-IDF без нормализации
results_no_norm = tfidf_search(query, field, idx, normalize=False)

# TF-IDF с нормализацией
results_norm = tfidf_search(query, field, idx, normalize=True)

print(f"Запрос №{query_id}: {query['text'][:70]}...")
print(f"Слов в запросе: {len(query[field])}")

print(f"\n{'Без нормализации':-^50}")
print(f"{'Doc ID':<10} {'Score':<12}")
print("-" * 22)

for doc_id, score in results_no_norm[:10]:
    print(f"{doc_id:<10} {score:<12.6f}")

print(f"\n{'С нормализацией':-^50}")
print(f"{'Doc ID':<10} {'Score':<12}")
print("-" * 22)

for doc_id, score in results_norm[:10]:
    print(f"{doc_id:<10} {score:<12.6f}")


if query_id in relevance:
    relevant_set = set(relevance[query_id])
    top10_no_norm = set(d for d, _ in results_no_norm[:10])
    top10_norm = set(d for d, _ in results_norm[:10])
    
    print(f"\nРелевантных: {len(relevant_set)}")
    print(f"Без нормализации: Precision@(K=10) = {len(top10_no_norm & relevant_set) / 10:.4f}")
    print(f"С нормализацией:   Precision@(K=10) = {len(top10_norm & relevant_set) / 10:.4f}")

## Нормализация и формулы

Здесь TF-IDF — сумма `qtf(t) * tf(t,d) * ln(N/df(t))`, при `normalize=True` разделённая на число обработанных токенов документа. Это базовый взвешенный скор, а не косинусная близость двух TF-IDF-векторов. Деление на длину может чрезмерно повышать короткие документы.

BM25 использует положительный сглаженный IDF `ln(1 + (N-df+0.5)/(df+0.5))` и насыщение TF `tf*(k1+1)/(tf+k1*(1-b+b*dl/avgdl))`. Повторения термина в запросе учитываются линейно в обеих моделях. Дополнительное деление BM25 на длину не применяется: нормализация уже включена в формулу.

## Поиск по полям и fallback

Для любого выбранного поля документа используется весь текст запроса с тем же методом обработки. Отсутствие заголовка или автора в записи запроса не делает поиск по заголовкам или авторам документов пустым.

Fallback включается явно параметром `fallback_idx` или `fallback_field`, когда первичная выдача пуста. Обработка запроса соответствует запасному индексу, а ранжированные скоры умножаются на 0.5. Этот множитель не меняет порядок и не является вероятностью релевантности. Основное сравнение полей выполняется без fallback.

In [ ]:
def bm25_search(query, query_field, rich_idx, k1=1.2, b=0.75, fallback_idx=None):
    if not math.isfinite(k1) or k1 <= 0 or not math.isfinite(b) or not 0 <= b <= 1:
        raise ValueError("BM25 requires finite k1 > 0 and 0 <= b <= 1")
    scores = defaultdict(float)
    avg_dl = rich_idx["avg_dl"]
    if avg_dl > 0:
        for word in query[query_field]:
            for doc_id, meta in rich_idx["inverted_index"].get(word, {}).items():
                tf = meta["tf"]
                dl = rich_idx["doc_lengths"][doc_id]
                scores[doc_id] += rich_idx["bm25_idf"][word] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avg_dl))
    if not scores and fallback_idx is not None:
        results = bm25_search(query, query_field_for(fallback_idx["field"]), fallback_idx, k1, b)
        return [(doc_id, score * 0.5) for doc_id, score in results]
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))

In [ ]:
def precision_at_k(results, relevant, k):
    if k < 0:
        raise ValueError("k must be nonnegative")
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & set(relevant)) / k if k else 0.0

def recall_at_k(results, relevant, k):
    if k < 0:
        raise ValueError("k must be nonnegative")
    relevant = set(relevant)
    top_k = {doc_id for doc_id, _ in results[:k]}
    return len(top_k & relevant) / len(relevant) if relevant else 0.0

def average_precision(results, relevant):
    relevant = set(relevant)
    hits = 0
    total = 0.0
    seen = set()
    for position, (doc_id, _) in enumerate(results, 1):
        if doc_id in relevant and doc_id not in seen:
            hits += 1
            total += hits / position
        seen.add(doc_id)
    return total / len(relevant) if relevant else 0.0

In [ ]:
import random

judged_queries = sorted(qid for qid in queries if relevance.get(qid))
test_queries = sorted(random.Random(52).sample(judged_queries, 5))
ks = [1, 3, 5, 10]
rows = []
for model, search in (("TF-IDF", tfidf_search), ("BM25", bm25_search)):
    for field in fields:
        for qid in judged_queries:
            results = search(queries[qid], query_field_for(field), rich_indexes[field])
            relevant = relevance[qid]
            row = {"model": model, "field": field, "query": qid, "AP": average_precision(results, relevant)}
            for k in ks:
                row[f"P@{k}"] = precision_at_k(results, relevant, k)
                row[f"R@{k}"] = recall_at_k(results, relevant, k)
            rows.append(row)
evaluation = pd.DataFrame(rows)
summary = evaluation.groupby(["model", "field"])[["AP"] + [metric for k in ks for metric in (f"P@{k}", f"R@{k}")]].mean().rename(columns={"AP": "MAP"})
print(f"Evaluation: {len(judged_queries)} judged queries of {len(queries)}; no fallback; macro averages.")
display(summary)
print(f"Illustrative random sample: {test_queries}")
display(evaluation[(evaluation["field"] == "stemmed_words") & evaluation["query"].isin(test_queries)])

boolean_rows = []
for operator in ("OR", "AND"):
    for qid in judged_queries:
        found = boolean_search(queries[qid], "stemmed_words", "stemmed_words", operator)
        relevant = relevance[qid]
        hits = len(found & relevant)
        boolean_rows.append({"operator": operator, "precision": hits / len(found) if found else 0.0,
                             "recall": hits / len(relevant), "documents": len(found)})
boolean_summary = pd.DataFrame(boolean_rows).groupby("operator").mean()
display(boolean_summary)

## Итоги

Основной результат исследования — таблица средних метрик `summary` по всем запросам с разметкой. Пять случайных запросов служат иллюстрацией и не доказывают преимущество алгоритма на всей коллекции. Значения P@K, R@K и MAP могут отдавать предпочтение разным конфигурациям.

Сравниваются сохранение стоп-слов (`raw_words`), очистка (`tokens`), стемминг и настоящая лемматизация WordNet. Porter приводит `library` и `libraries` к `librari`, но `librarianship` — к `librarianship`: это не один стем.

Выбор модели и поля следует делать по вычисленным метрикам с учётом цели поиска. Параметры BM25 `k1=1.2, b=0.75` фиксированы; оптимальность параметров не доказана, отдельной валидационной выборки нет. Разметка CISI не покрывает все запросы, а результаты нельзя автоматически переносить на другие коллекции.

Формулы и принципы оценки: [Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html).